# Qwen3.8-27B abliterated (uncensored) — Colab L4

**用前先确认运行时是 L4**：代码执行程序 → 更改运行时类型 → L4 GPU（22.5GB 显存，1.71 CU/小时）。

跑完这个 notebook 会得到：
1. 一个能对话的 Ollama 模型 `qwen27b-unc`
2. 一个公网 OpenAI 兼容端点（Cloudflare 隧道），任何客户端都能接



In [ ]:
!nvidia-smi
import torch, subprocess
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print(subprocess.run(['free','-g'],capture_output=True,text=True).stdout)


In [ ]:
# 装 Ollama
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > /tmp/ollama.log 2>&1 &
import time; time.sleep(10)
!curl -s http://127.0.0.1:11434/api/tags


In [ ]:
# 下载 GGUF（Q4_K_M ≈ 19GB）。hf_transfer 多线程，几分钟
!pip -q install -U huggingface_hub hf_transfer
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER']='1'
from huggingface_hub import hf_hub_download, list_repo_files

repo='huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF'
files=[f for f in list_repo_files(repo) if f.endswith('.gguf') and 'Q4_K_M' in f]
print('候选文件:', files)
path=hf_hub_download(repo_id=repo, filename=files[0], local_dir='/content/models')
print('下载完成:', path, round(os.path.getsize(path)/1e9,2), 'GB')


In [ ]:
# 建模型：上下文压到 8K，L4 22.5GB 才装得下
with open('/tmp/Modelfile','w') as f:
    f.write('FROM /content/models/' + __import__('os').path.basename(path) + '\nPARAMETER num_ctx 8192\nPARAMETER temperature 0.7\n')
!ollama create qwen27b-unc -f /tmp/Modelfile
!ollama list
!curl -s http://127.0.0.1:11434/api/chat -d '{"model":"qwen27b-unc","messages":[{"role":"user","content":"用一句话介绍你自己"}],"stream":false}' | head -c 500


In [ ]:
# 开公网隧道，拿到 OpenAI 兼容地址
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!nohup cloudflared tunnel --url http://127.0.0.1:11434 --no-autoupdate > /tmp/cf.log 2>&1 &
import time, re
time.sleep(15)
log=open('/tmp/cf.log').read()
m=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log)
print('='*60)
if m:
    print('公网 base_url:', m.group(0) + '/v1')
    print('api_key: 随便填一个字符串')
    print('model: qwen27b-unc')
else:
    print(log[-800:])
print('='*60)


In [ ]:
# 保持运行：这个 cell 别关，关掉 Colab 会在闲置后回收机器
import time
while True:
    time.sleep(300)
    print('alive', time.strftime('%H:%M'))
